#Intialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim,col

In [0]:
rename_map = {
    'ID': 'category_id',
    'CAT': 'category',
    'SUBCAT': 'subcategory',
    'MAINTENANCE': 'maintenance'
}

#Reading Data From Bronze Layer

In [0]:
df = spark.table('workspace.bronze.erp_px_cat_g1v2')

#Data Cleaning

##Trimming String columns

In [0]:
for column in df.dtypes:
    if column[1] == 'string':
        df = df.withColumn(column[0],trim(col(column[0])))

##Standardization

###Value Mapping (e.g. AC_BR → AC-BR)

In [0]:
df = df.withColumn('ID',F.regexp_replace(df['ID'],'_','-'))

###Column Naming Standardization

In [0]:
for old_name,new_name in rename_map.items():
    df = df.withColumnRenamed(old_name,new_name)

#Loading Data Into Silver Layer

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('silver.erp_px_category')